# Three-species hippocampus data preparation

This notebook prepares the human, pig, and rhesus macaque hippocampus data from GEO series **GSE186538**. The annotation table is matched by the `cell_name` barcode, and the final label is stored in `obs['cell_type']`.

## Expected directory layout

Keep the scripts and the small annotation CSV in this vignette. Large expression files may remain in `D:\\111icde_addition_experiments\\3_species_hippocampus`; they do not need to be copied into the repository.

The conversion script contains configurations for all three species. If the human matrix is absent, Human is reported and skipped automatically.

In [ ]:
from pathlib import Path

VIGNETTE_DIR = Path.cwd()
if VIGNETTE_DIR.name != "3_species_hippocampus":
    VIGNETTE_DIR = Path(r"D:\111icde_addition_experiments\MetaGeneFormer\Vignettes\3_species_hippocampus")

EXTERNAL_DATA_DIR = Path(r"D:\111icde_addition_experiments\3_species_hippocampus")
ANNOTATION_CSV = VIGNETTE_DIR / "data" / "three_species_hippocampus_barcode_cell_type.csv"
OUTPUT_DIR = EXTERNAL_DATA_DIR / "processed_h5ad"

print("Vignette:", VIGNETTE_DIR)
print("Input data:", EXTERNAL_DATA_DIR)
print("Annotation:", ANNOTATION_CSV)
print("Output:", OUTPUT_DIR)

## Option A: use the existing large files directly

This is the recommended option when the files already exist in the external data directory.

In [ ]:
expected = {
    "human": [
        "GSE186538_Human_cell_meta.txt.gz",
        "GSE186538_Human_counts.mtx.gz",
        "GSE186538_Human_genes.txt.gz",
    ],
    "pig": [
        "GSE186538_Pig_cell_meta.txt.gz",
        "GSE186538_Pig_counts.mtx.gz",
        "GSE186538_Pig_genes.txt.gz",
    ],
    "macaM": [
        "GSE186538_Rhesus_cell_meta.txt.gz",
        "GSE186538_Rhesus_counts.mtx.gz",
        "GSE186538_Rhesus_genes.txt.gz",
    ],
}

for species, filenames in expected.items():
    missing = [name for name in filenames if not (EXTERNAL_DATA_DIR / name).exists()]
    print(species, "ready" if not missing else f"missing: {missing}")

## Option B: download files from GEO

GEO provides all nine processed files. The Human count matrix is approximately 1.4 GB compressed. To download only the two currently required species, use `--species pig macaM`.

In [ ]:
# Run only when files need to be downloaded.
# This command downloads Pig and Rhesus without downloading the large Human matrix.
# !python "{VIGNETTE_DIR / 'download_three_species_hippocampus_data.py'}" --data-dir "{EXTERNAL_DATA_DIR}" --species pig macaM

To download all three species, including Human, remove the `--species` argument:

```powershell
python download_three_species_hippocampus_data.py --data-dir D:\111icde_addition_experiments\3_species_hippocampus
```

## Convert available species to H5AD

The script keeps only barcodes present in the annotation CSV, applies cell and gene QC, and writes `cell_type` into `obs`. Missing input matrices are skipped unless `--require-all` is supplied.

The following command is intentionally commented out so opening this notebook does not start the large conversion automatically.

In [ ]:
# Convert only Pig and Rhesus (output names: pig.h5ad and macaM.h5ad).
# !python "{VIGNETTE_DIR / 'prepare_three_species_hippocampus_h5ad.py'}" --input-dir "{EXTERNAL_DATA_DIR}" --annotation-csv "{ANNOTATION_CSV}" --output-dir "{OUTPUT_DIR}" --species pig macaM

When `GSE186538_Human_counts.mtx.gz` becomes available, run all configured species by omitting `--species`. The outputs are `human.h5ad`, `pig.h5ad`, and `macaM.h5ad`.

In [ ]:
# Optional validation after conversion.
# import anndata as ad
# for path in sorted(OUTPUT_DIR.glob('*.h5ad')):
#     adata = ad.read_h5ad(path, backed='r')
#     print(path.name, adata.shape, 'cell_type' in adata.obs.columns, adata.obs_names.is_unique)
#     adata.file.close()